# Instagram Engagement Prediction & Keyword Recommendation
This notebook demonstrates how to use BERT embeddings, TF-IDF, and XGBoost to predict engagement and recommend keywords for Instagram posts. It includes data upload, preprocessing, model training, evaluation, and visualization steps.

In [ ]:
# Install required packages (uncomment if running in Colab)
# !pip install transformers xgboost nltk scikit-learn matplotlib pandas torch

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBRegressor
from nltk.tokenize import word_tokenize
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import nltk
nltk.download('punkt')

In [ ]:
# Colab file upload utility
from google.colab import files
uploaded = files.upload()
# After upload, use the filename as needed, e.g., 'final_combined_dataset.csv'

In [ ]:
# Initialize BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model.to(device)
bert_model.eval()

In [ ]:
# Function to get BERT embeddings
def get_bert_embeddings(texts, max_length=128, batch_size=16):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size].tolist() if isinstance(texts, pd.Series) else texts[i:i + batch_size]
        inputs = tokenizer(batch_texts, return_tensors='pt', max_length=max_length, padding=True, truncation=True)
        inputs = {key: val.to(device) for key, val in inputs.items()}
        with torch.no_grad():
            outputs = bert_model(**inputs)
        batch_embeddings = outputs.pooler_output.cpu().numpy().astype(np.float32)
        embeddings.append(batch_embeddings)
    return np.vstack(embeddings)

In [ ]:
# Load dataset
data = pd.read_csv('final_combined_dataset.csv')
print(f"Loaded dataset with {data.shape[0]} rows and {data.shape[1]} columns.")
data.head()

In [ ]:
# Preprocessing: Handle missing values
data['caption'] = data['caption'].fillna('')
data['comment_text'] = data['comment_text'].fillna('')
data = data.fillna(0)

In [ ]:
# Aggregate comment features per post
def aggregate_comment_features(data):
    comment_features = data.groupby('post_id').agg({
        'comment_text': lambda x: ' '.join(x),
        'avg_comment_sentiment': 'mean',
        'sentiment_weighted_engagement': 'first'
    }).reset_index()
    comment_bert_embeddings = get_bert_embeddings(comment_features['comment_text'])
    comment_bert_cols = [f'comment_bert_{i}' for i in range(comment_bert_embeddings.shape[1])]
    comment_bert_df = pd.DataFrame(comment_bert_embeddings, columns=comment_bert_cols)
    comment_features = pd.concat([
        comment_features[['post_id', 'avg_comment_sentiment', 'sentiment_weighted_engagement']],
        comment_bert_df
    ], axis=1)
    return comment_features

In [ ]:
# Aggregate comment features and merge with original data
comment_features = aggregate_comment_features(data)
data = data.drop_duplicates(subset=['post_id']).merge(comment_features, on='post_id', suffixes=('', '_agg'))

In [ ]:
# Fit TF-IDF vectorizer for captions
tfidf_caption = TfidfVectorizer(max_features=100, stop_words='english')
tfidf_caption.fit(data['caption'])

In [ ]:
# Feature extraction
numerical_cols = ['avg_comment_sentiment']
scaler = StandardScaler()
X_numerical = scaler.fit_transform(data[numerical_cols]).astype(np.float32)

# Caption and comment BERT embeddings
caption_bert_embeddings = get_bert_embeddings(data['caption'])
comment_bert_embeddings = get_bert_embeddings(data['comment_text'])

# Combine features
X_combined = np.hstack([
    X_numerical,
    caption_bert_embeddings,
    comment_bert_embeddings
]).astype(np.float32)

y = data['sentiment_weighted_engagement'].astype(np.float32)

In [ ]:
# Train-test split
if len(data) < 2:
    print("Insufficient data to train the model.")
else:
    X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)

In [ ]:
# Train XGBoost model
xgb_model = XGBRegressor(objective='reg:squarederror', n_estimators=50, learning_rate=0.1, max_depth=5)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

In [ ]:
# Evaluate model
def evaluate_model(y_true, y_pred, model_name):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name} Performance:")
    print(f"MSE: {mse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")
    return mse, mae, r2

xgb_metrics = evaluate_model(y_test, xgb_pred, "XGBoost")

In [ ]:
# Visualize predictions vs actual
plt.figure(figsize=(8,5))
plt.scatter(y_test, xgb_pred, alpha=0.6)
plt.xlabel('Actual Engagement')
plt.ylabel('Predicted Engagement')
plt.title('Actual vs Predicted Engagement (XGBoost)')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.show()

In [ ]:
# Visualize feature importances
importances = xgb_model.feature_importances_
plt.figure(figsize=(10,4))
plt.bar(range(len(importances)), importances)
plt.title('XGBoost Feature Importances')
plt.xlabel('Feature Index')
plt.ylabel('Importance')
plt.show()

In [ ]:
# Recommend keywords
def recommend_keywords(data, model, scaler, tfidf_vectorizer, top_n=5, max_candidates=100):
    avg_comment_sentiment = data['avg_comment_sentiment'].mean()
    recent_comment_text = data['comment_text'].iloc[-1] if not data['comment_text'].empty else ""
    numerical_features = np.array([avg_comment_sentiment]).reshape(1, -1)
    numerical_scaled = scaler.transform(numerical_features).astype(np.float32)
    candidate_words = list(tfidf_vectorizer.vocabulary_.keys())[:max_candidates]
    comment_bert_base = get_bert_embeddings([recent_comment_text])[0].astype(np.float32)
    scores = []
    for word in candidate_words:
        caption = word
        caption_bert = get_bert_embeddings([caption])[0].astype(np.float32)
        input_features = np.hstack([
            numerical_scaled,
            caption_bert,
            comment_bert_base
        ]).astype(np.float32)
        score = model.predict(input_features)[0]
        scores.append((word, score))
    scores.sort(key=lambda x: x[1], reverse=True)
    recommendations = [word for word, score in scores[:top_n]]
    return recommendations

In [ ]:
# Evaluate recommendations
def evaluate_recommendations(data, model, scaler, tfidf_vectorizer, top_n=5):
    actual_engagements = []
    predicted_engagements = []
    recommendations_list = []
    test_data = data.iloc[-1:]
    for idx, row in test_data.iterrows():
        caption = row['caption']
        comment_text = row['comment_text']
        numerical_features = np.array([row['avg_comment_sentiment']]).reshape(1, -1)
        actual_engagement = row['sentiment_weighted_engagement']
        recommendations = recommend_keywords(data, model, scaler, tfidf_vectorizer, top_n)
        modified_caption = " ".join(recommendations)
        caption_bert = get_bert_embeddings([modified_caption])[0].astype(np.float32)
        comment_bert = get_bert_embeddings([comment_text])[0].astype(np.float32)
        numerical_scaled = scaler.transform(numerical_features).astype(np.float32)
        input_features = np.hstack([
            numerical_scaled,
            caption_bert,
            comment_bert
        ]).astype(np.float32)
        pred_engagement = model.predict(input_features)[0]
        actual_engagements.append(actual_engagement)
        predicted_engagements.append(pred_engagement)
        recommendations_list.append(recommendations)
    mse = mean_squared_error(actual_engagements, predicted_engagements)
    mae = mean_absolute_error(actual_engagements, predicted_engagements)
    r2 = r2_score(actual_engagements, predicted_engagements)
    print("Recommendation Performance:")
    print(f"MSE: {mse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")
    print("\nSample Recommendations:")
    for i, (row, recs) in enumerate(zip(test_data.itertuples(), recommendations_list)):
        print(f"Post {i+1}:")
        print(f"  Original Caption: {row.caption}")
        print(f"  Comment Text: {row.comment_text[:100]}...")
        print(f"  Recommended Keywords: {recs}")
        print(f"  Actual Engagement: {row.sentiment_weighted_engagement:.4f}")
        print(f"  Predicted Engagement: {predicted_engagements[i]:.4f}\n")
    return mse, mae, r2, recommendations_list

In [ ]:
# Main execution
recommended_keywords = recommend_keywords(data, xgb_model, scaler, tfidf_caption)
print(f"Recommended Keywords for Next Post: {recommended_keywords}")

# Evaluate recommendations
mse, mae, r2, recommendations = evaluate_recommendations(data, xgb_model, scaler, tfidf_caption)

## Summary
- The notebook loads Instagram post/comment data, extracts features using BERT and TF-IDF, and trains an XGBoost regressor.
- It visualizes model performance and feature importances.
- It recommends keywords for future posts and evaluates their predicted impact on engagement.
- You can upload your own dataset using the upload cell at the top.